In [0]:
%pip install google-api-python-client --quiet
dbutils.library.restartPython()

In [0]:
from googleapiclient.discovery import build

api_key = dbutils.secrets.get(scope="youtube-scope", key="youtube-api-key")

youtube = build("youtube", "v3", developerKey=api_key)

In [0]:
response = youtube.channels().list(
    part="snippet,statistics",
    id="UCEGvwrlFT1Ml4VJASkA9rMA"
).execute()

print(response)

In [0]:
import json
from pyspark.sql.functions import from_json, schema_of_json, col, lit

json_str = json.dumps(response)
df = spark.createDataFrame([(json_str,)], ["value"])
schema = schema_of_json(lit(json_str))
df = df.select(from_json(col("value"), schema).alias("data")).select("data.*")
df.display()

In [0]:
from pyspark.sql.functions import current_timestamp

df = df.withColumn("ingestion_time", current_timestamp())

In [0]:
df.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("workspace_youtube_1.bronze.raw_channels")